# Question 6: Error analysis

Which characteristics of the sentence pairs predict whether a system gets them right and do these differ between our major design choices?

We use the predictions saved during crossvalidation: every pair was predicted once, by the model of the fold where it was in the validation set. So for each system we know for all 57,401 pairs whether it was correct, without training anything new.

Systems:
- `pool_max`: Question 3 embed-and-pool with max pooling with no word order
- `lstm_base`: Question4 siamese BiLSTM on frozen GloVe
- `lstm_joint`: Question 4 one BiLSTM reading "sentence1 [SEP] sentence2"
- `lstm_best`: Question 4 best configuration (finetuned GloVe, hidden size 256, dropout 0.3)


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

from src.utils import load_datasets

train_df, dev_df, _ = load_datasets()
df = pd.concat([train_df, dev_df]).reset_index(drop=True)[["id", "sentence1", "sentence2", "label"]]

q3 = json.loads(Path("results/q3_glove.json").read_text())
q4 = json.loads(Path("results/q4_glove.json").read_text())

SYSTEMS = {
    "pool_max": q3["pool_max"],   
    "lstm_base": q4["base"],      
    "lstm_joint": q4["joint"],    
    "lstm_best": q4["combined"],  
}


folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=64).split(df, df["label"]))
df["fold"] = -1
for k, (_, val_idx) in enumerate(folds):
    df.loc[val_idx, "fold"] = k

for name, r in SYSTEMS.items():
    preds = np.array(r["predictions"])
    assert len(preds) == len(df) and (preds >= 0).all(), name
    df[f"pred_{name}"] = preds
    df[f"correct_{name}"] = (preds == df["label"]).astype(int)

    # accuracies recomputed per fold must equal the saved ones
    recomputed = [df.loc[val_idx, f"correct_{name}"].mean() for _, val_idx in folds]
    assert np.allclose(recomputed, r["accuracies"]), f"{name}: predictions do not match the pairs"

print(f"{len(df)} pairs, alignment check passed for {list(SYSTEMS)}")
display(df[[f"correct_{n}" for n in SYSTEMS]].mean().mul(100).round(2).rename("accuracy (%)"))
display(df.head())


57401 pairs, alignment check passed for ['pool_max', 'lstm_base', 'lstm_joint', 'lstm_best']


correct_pool_max      56.41
correct_lstm_base     84.28
correct_lstm_joint    71.19
correct_lstm_best     87.83
Name: accuracy (%), dtype: float64

,id,sentence1,sentence2,label,fold,pred_pool_max,correct_pool_max,pred_lstm_base,correct_lstm_base,pred_lstm_joint,correct_lstm_joint,pred_lstm_best,correct_lstm_best
0,1,"In Paris , in October 1560 , he secretly met t...","In October 1560 , he secretly met with the Eng...",0,1,0,1,0,1,0,1,0,1
1,2,The NBA season of 1975 -- 76 was the 30th seas...,The 1975 -- 76 season of the National Basketba...,1,1,1,1,1,1,1,1,1,1
2,3,"There are also specific discussions , public p...","There are also public discussions , profile sp...",0,3,1,0,0,1,0,1,0,1
3,4,When comparable rates of flow can be maintaine...,The results are high when comparable flow rate...,1,3,1,1,0,0,0,0,0,0
4,5,It is the seat of Zerendi District in Akmola R...,It is the seat of the district of Zerendi in A...,1,1,0,0,1,1,1,1,1,1
